In [1]:
import pandas as pd
import os
import glob
from datetime import datetime

In [2]:
def convert_txt_to_csv(input_folder, output_folder):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    txt_files = glob.glob(os.path.join(input_folder, "*.txt"))

    for file_path in txt_files:
        file_name = os.path.basename(file_path)
        interval_rows = []
        packet_buffer = []
        global_start_time = None  # Reference for relative timing
        
        with open(file_path, 'r') as f:
            for line in f:
                line = line.strip()
                if not line: continue
                
                try:
                    parts = line.split(', ')
                    raw_ts = parts[0]
                    raw_data = parts[2].split(':', 4)
                    while len(raw_data) < 5: raw_data.append("")
                    
                    ts_obj = datetime.strptime(raw_ts, '%Y-%m-%d %H:%M:%S.%f')
                    
                    # Set the file's "Zero Hour"
                    if global_start_time is None:
                        global_start_time = ts_obj

                    current_packet = {
                        'ts': ts_obj,
                        'id': raw_data[0],
                        'sync': raw_data[2],
                        'ack': raw_data[3],
                        'is_data': bool(raw_data[4].strip())
                    }

                    # Trigger interval processing on Sync flag
                    if current_packet['sync'] == '1' and packet_buffer:
                        metrics = calculate_interval_metrics(packet_buffer, global_start_time)
                        if metrics:
                            interval_rows.append(metrics)
                        packet_buffer = []

                    packet_buffer.append(current_packet)
                    
                except Exception as e:
                    print(f"Error in {file_name}: {e}")

        if interval_rows:
            df = pd.DataFrame(interval_rows)
            df.to_csv(os.path.join(output_folder, file_name.replace(".txt", "_intervals.csv")), index=False)

def calculate_interval_metrics(buffer, global_ref):
    """Calculates stats for a single sync-to-sync window."""
    
    # 1. Message Type Counts
    data_packets = [p for p in buffer if p['is_data']]
    ack_count = sum(1 for p in buffer if p['ack'] == '1')
    
    # NEW REQUIREMENT: Discard if no data OR no acks
    if not data_packets or ack_count == 0:
        return None

    sync_count = sum(1 for p in buffer if p['sync'] == '1')
    data_count = len(data_packets)

    # 2. Relative Timestamps (Seconds from start of file)
    rel_start = (buffer[0]['ts'] - global_ref).total_seconds()
    rel_end = (buffer[-1]['ts'] - global_ref).total_seconds()
    interval_duration = rel_end - rel_start

    # 3. Empty Timeslots (ID Sequence Check: 01 -> 02 -> 03)
    missing_slots = 0
    id_map = {"01": 1, "02": 2, "03": 3}
    
    for i in range(len(buffer) - 1):
        curr_id_val = id_map.get(buffer[i]['id'])
        next_id_val = id_map.get(buffer[i+1]['id'])
        
        if curr_id_val and next_id_val:
            # Expected sequence is 1->2, 2->3, 3->1
            expected_next = (curr_id_val % 3) + 1
            if next_id_val != expected_next:
                # Count gaps (e.g., if 01 followed by 03, missing_slots += 1)
                missing_slots += 1

    # 4. Data Packet Timing (Max/Min gap between data payloads)
    data_intervals = []
    if len(data_packets) > 1:
        for i in range(len(data_packets) - 1):
            diff = (data_packets[i+1]['ts'] - data_packets[i]['ts']).total_seconds()
            data_intervals.append(diff)
    
    return {
        "Rel Start (s)": round(rel_start, 6),
        "Rel End (s)": round(rel_end, 6),
        "Interval Duration": round(interval_duration, 6),
        "Syncs": sync_count,
        "Acks": ack_count,
        "Data Packets": data_count,
        "Missing Slots": missing_slots,
        "Max Data Gap": max(data_intervals) if data_intervals else 0,
        "Min Data Gap": min(data_intervals) if data_intervals else 0
    }

In [3]:
# --- Execution ---
# Replace 'input_data_folder' with the path to your folder containing .txt files
# input_folder = 'Test Files' 
input_folder = '../../../Secured-Network-EH-TDMA/TDMA_Tooling-main/' 
output_folder = 'Converted CSVs'

convert_txt_to_csv(input_folder, output_folder)

Error in Trojan_100_4_170ms.txt: list index out of range
Error in Trojan_100_4_170ms.txt: list index out of range
Error in Trojan_100_4_170ms.txt: list index out of range
Error in Trojan_100_4_170ms.txt: list index out of range
Error in Burst_40_3_120s.txt: list index out of range
Error in Burst_40_3_120s.txt: list index out of range
Error in Burst_40_3_120s.txt: list index out of range
Error in Burst_40_4.txt: list index out of range
Error in Burst_40_4.txt: list index out of range
Error in Burst_40_4.txt: list index out of range
Error in Burst_40_4.txt: list index out of range
Error in Burst_40_1_10s.txt: list index out of range
Error in Burst_40_1_10s.txt: list index out of range
Error in Burst_40_1_10s.txt: list index out of range
Error in Burst_40_1_10s.txt: list index out of range
Error in Burst_40_1_10s.txt: list index out of range
Error in Burst_40_1_10s.txt: list index out of range
Error in Normal_40_3.txt: list index out of range
Error in Normal_40_3.txt: list index out of ra